# Levantamento das features

## Features de Satisfação

### **Feature 1: SatisfactionScore**

**Justificativa**: 

Utilizar variáveis de satisfação para retornar um valor geral satisfatório entre o funcionário e a empresa. Cada uma mede um aspecto diferente, mas nenhuma delas separadamente diz que no geral o funcionário está satisfeito ou insatisfeito com a empresa. 

Para essa abordagem utilizaremos JobSatisfaction, EnvironmentSatisfaction e RelationshipSatisfaction. Temos a opção de adicionarmos WorkLifeBalance, porém essa variável mede o equilíbrio entre a vida pessoal e o trabalho, portanto o conceio pode ser parecido mas são conceitos diferentes.

Poderíamos adicionar uma feature a mais para medir o relacionamento do funcionário com a vida pessoal, porém, a correlação entre as duas seriam altas e teríamos uma multicolineariedade, caso as variáveis fossem selecionadas para modelagem. Tomamos a decisão de manter apenas as três variáveis que medem a satisfação do funcionário em relação a empresa.

- JobSatisfaction            ->  satisfação com o cargo
- EnvironmentSatisfaction    ->  satisfação com o ambiente de trabalho
- RelationshipSatisfaction   ->  satisfação com os relacionamentos (colegas/gestor)

**Calculo:** média de (JobSatisfaction + EnvironmentSatisfaction + RelationshipSatisfaction) / 3 


### **Feature 2: CriticalSatisfactionFlag**

**Justificativa**: Criar um indicador binário de satisfação crítica. Com isso facilita o modelo em separar os funcionários com risco maior de attrito. O threshold de 2 foi escolhido para identificar somente quem está insatisfeito (insatisfação baixa e média).

**Cálculo**: 1 se SatisfactionScore <= 2, senão 0


## Features de Carreira e Tempo

### **Feature 3: RatioCareerCompany**

**Justificativa**:  

Mede qual foi a proporção de anos trabalhados na empresa TechCorp Brasil em relação a quantidade total de anos trabalhados. Essa feature pode identificar funcionários que fizeram a sua carreira dentro da empresa TechCorp Brasil (mais difícil de entrar em attrition). A feature também pode encontrar funcionários que mudaram de emprego muitas vezes (mais fácil de entrar em attrition) e assim o modelo pode conseguir identificar padrões e realizar previsões mais próximas da realidade.

- Valor alto (ex: 0.90) = fez carreira aqui, mais estável
- Valor baixo (ex: 0.20) = passou por muitas empresas, mais fácil de sair da empresa
- Adicionamos +1 ao denominador para evitar divisão por zero

**Cálculo:** YearsAtCompany / (TotalWorkingYears + 1)

### **Feature 4: RoleStagnation**

**Justificativa**:
Mede o tempo que o funcionário está no mesmo cargo em relação ao tempo total que ele tem trabalhando na empresa. Essa feature consegue identificar os funcipnários que possuem muitos anos em uma mesma função. Sabemos que um funcionário que fica muitos anos em um mesmo cargo, pode gerar insatisfação e não contentamento com o seu trabalho e pode acabar buscando por outras oportunidades fora da empresa.

- valor próximo de 1.0 = nunca mudou de cargo
- valor baixo = mudou de cargo recentemente

Cálculo: YearsInCurrentRole / (YearsAtCompany + 1)


### **Feature 5: YearsWithoutPromotion**

**Justificativa**:
Mede quanto tempo o funcionário está sem promoção em relação ao tempo total de anos trabalhados na empresa. Ao realizar o levantamento dessa feature, surgiu a dúvida de estar fortemente correlacionada com a **Feature 2 RoleStagnation**, porém, são medidas diferentes. Aqui estamos medindo a promoção de um nível atual para um nível acima, por exemplo: Um funcionário que está a muito tempo como Pleno (mid-level), tende a estar insatisfeito com a situação atual e pode procurar oportunidades em um nível acima em outra empresa, pois não vê oportunidades de crescimento na empresa atual.

- valor alto = estagnado na carreira, possível frustração e aberto a mudanças fora da empresa.
- valor baixo = promovido recentemente, mais engajado com o trabalho atual.

**Cálculo:** YearsSinceLastPromotion / (YearsAtCompany + 1)

### **Feature 6: AgeGroup**

**Justificativa**: Transformar a variável contínua idade em categorias (Jovem, Adulto, Senior). Pensando no mercado de trabalho real temos algumas hipóteses que pode fazer muito sentido com o nosso contexto:

Jovem com menos de 30 anos:
- Início de carreira, buscando crescimento rápido
- Menos "preso" à empresa (menos responsabilidades financeiras)
- Mais aberto a trocar de emprego por oportunidades ainda melhores
- Tem maior mobilidade

Adulto entre 31 e 45 anos:
- Meio de carreira, buscando estabilidade
- Geralmente tem família, casa, responsabilidade financeira, filhos
- Trocar de emprego pode ser mais arriscado mas é possível que aconteça
- Comportamento moderado

Senior com mais de 45 anos:
- Carreira consolidada
- É mais difícil de se realocar no mercado de trabalho
- Valorizza estabilidade e benefícios
- Tende a buscar maior estabilidade

Jovens tendem a ter mais mobilidade enquanto seniores buscam por estabilidade.



## Features de Compensação

### **Feature 7: IncomePerLevel**

**Justificativa**: Mede a comparação entre o salário com o nível hierárquico do funcionário. Se o salário for baixo para o seu nível, o funcionário pode sentir-se desvalorizado e tem mais risco de entrar em attrition, pois a expectativa é que o salário esteja de acordo com o nível do funcionário.

**Exemplo:** Ana tem um salário de $3.000e está em JobLevel 1. Bruno tem um salário de $3.000 e está em JobLevel 3. Realizando o os cálculos, temos o seguinte resultado.

| Funcionário | Salário | JobLevel | income_per_level | Interpretação |
|---|---|---|---|---|
|Ana | 3.000 |1|3.000 / 1 = 3.000| Bem paga para nível 1|
|Bruno | 3.000 | 3 | 3.000 / 3 = 1.000 | Mal pago para nível 3|

Bruno, tem mais risco de attrition, pois a expectativa salarial de do nível 3 é muito maior do que $3.000.


**Cálculo:** MonthlyIncome / JobLevel


### **Feature 8: IncomeMedianJob**

**Justificativa**: Mede o salário atual do funcionário em relação a mediana de salário dos pares e parceiros que estão no mesmo cargo. Um funcionário abaixo de 1.0 ganha menos que os pares e um funcionário acima de 1.0 ganha mais que os seus pares. Com isso, funcionários que ganham menos que os seus pares e parceiros de um mesmo cargo, tendem a ficarem insatisfeitos e aberto a mudanças fora da empresa.

**Calculo:** MonthlyIncome / mediana do MonthlyIncome por JobRole

## Feature Risco e Comportamento

### **Feature 9 - RiskOvertimeDistance**

**Justificativa:** Mede a interação do funcionário que faz hora extra e mora longe. Normalmente um funcionário que realiza hora extra e mora longe da sua casa, acumula dois fatores de desgaste simultaneamente. O primeiro fator é o desgaste mental do trabalho em fazer mais horas do que é proposto e também o desgaste físico de sair tarde do trabalho e fazer todo o percurso até sua casa. O desgaste físico pode estar relacionado ao trânsito pesado para voltar, trens, metrôs e onibus superlotados, horário de pico, entre outros fatores.

**Cálculo:** (OverTime_encoded) × DistanceFromHome

**Variável OverTime:** Valor binário Yes e No. Será necessário aplicar valores numéricos para realizar o cálculo. Iremos aplicar Yes = 1 e No = 0, chamando a variável de OverTime_encoded.


### **Feature 10 - RiskWorklifeOvertime**

**Justificativa:** Mede a interação entre o equilíbrio vida-trabalho ruim e hora extra. A feature captura que o risco é alto quando os dois fatores se acumulam, pois hora extra sozinha ou equilíbrio ruim sozinho não geram o mesmo efeito para que o modelo entenda a situação de risco do funcionário. A união entre as duas variáveis é muito boa para entender alguns cenários de risco entre a vida e o trabalho do funcionário.

**Cálculo:** (5 - WorkLifeBalance) × (OverTime_encoded)

**Exemplo**:

|WorkLifeBalance|OverTime|Cálculo|Resultado|Risco|
|---|---|---|---|---|
(Excelente)   | No       | (5-4) × 0 = 1×0  |    0      | Sem risco 
(Ruim)        | Yes      | (5-1) × 1 = 4×1  |    4      | Crítico
(Regular)     | Yes      | (5-2) × 1 = 3×1  |    3      | Alto
(Bom)         | Yes      | (5-3) × 1 = 2×1  |    2      | Médio
(Excelente)   | Yes      | (5-4) × 1 = 1×1  |    1      | Baixo

 Sem essa feature, o modelo veria duas variáveis separadas:
- WorkLifeBalance = 1
- OverTime = Yes

-> O modelo teria que aprender sozinho que a combinação dos dois é mais perigosa do que cada uma de forma isolada

Com essa feature, o modelo recebe direto:
- worklife_overtime_risk = 4

-> A informação de INTERAÇÃO já está pronta e facilita muito o trabalho do modelo

# Feature Enginnering

#### O que é Feature Engineering

- É a transformação de dados brutos em novas variáveis com o objetivo de aumentar a capacidade do modelo em realizar previsões. Para o nosso contexto, precisamos realizar a etapa de Feature Engineering para criarmos features que ajudem na identificação de padrões e que se correlacionem com a nossa target (Attrition), pois até o momento, nenhuma das variáveis contidas no dataset apresentou correlação relevante com a target, conforme evidenciado nas três camadas de análise da EDA gráfica, estatística e correlacional.